# Unit 4 Assignment: Evaluated Agentic RAG System

This notebook implements a full **self-evaluating agentic RAG pipeline** in one place:
1. Build a FAISS knowledge base
2. Agent 1 (RAG) answers with retrieved context
3. Agent 2 (Evaluator) scores faithfulness and relevancy via DeepEval
4. Agent 3 (Revisor) improves failed answers
5. Run on 5 in-domain + 2 adversarial questions

In [1]:
%pip install -q crewai langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers deepeval python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai-tools 1.14.2 requires tiktoken~=0.8.0, but you have tiktoken 0.12.0 which is incompatible.
litellm 1.83.12 requires openai==2.24.0, but you have openai 2.32.0 which is incompatible.
litellm 1.83.12 requires pydantic==2.12.5, but you have pydantic 2.11.10 which is incompatible.
litellm 1.83.12 requires python-dotenv==1.0.1, but you have python-dotenv 1.1.1 which is incompatible.
trulens-core 2.7.2 requires rich<14.0.0,>=13.6.0, but you have rich 14.3.4 which is incompatible.
trulens-dashboard 2.7.2 requires rich<14.0,>=13.6, but you have rich 14.3.4 which is incompatible.
trulens-providers-openai 2.7.2 requires openai<2.0.0,>=1.52.1, but you have openai 2.32.0 which is incompatible.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pyt

In [8]:
import os
import json
import re
import time
from textwrap import dedent

import pandas as pd
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not found. Add it to .env and rerun.")

os.environ["OPENAI_API_KEY"] = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

llm = LLM(
    model="groq/llama-3.1-8b-instant",
    temperature=0.2,
    api_key=GROQ_API_KEY
)

print("Environment and libraries loaded.")

Environment and libraries loaded.


## Part 1: Knowledge Base

**Chosen topic:** Renewable energy systems and grid modernization.

Why this topic: it has technical detail, policy context, and enough diverse facts to test both retrieval quality and hallucination resistance.

In [4]:
knowledge_base_text = dedent("""
Renewable energy systems combine technologies, market mechanisms, and grid operations.
Solar photovoltaic (PV) generation converts sunlight directly into electricity. The cost of utility-scale
solar fell dramatically over the last decade due to module manufacturing scale, efficiency gains, and lower
financing costs. Wind power converts kinetic energy from air movement using turbines and has become a major
source of new generation in many countries. Onshore wind is typically cheaper than offshore wind, while
offshore wind can provide stronger and steadier wind resources near coastal demand centers.

A modern power grid must keep supply and demand balanced in real time. Historically, this was done mostly
with dispatchable fossil-fuel plants. As variable renewable energy increases, balancing requires additional
flexibility from storage, transmission, demand response, and fast-ramping generation. Lithium-ion batteries
are widely used for short-duration storage, often around one to four hours, and are valuable for frequency
control, reserve services, and shifting solar output from midday to evening peaks. Pumped hydro storage can
provide longer duration storage where geography allows.

Transmission expansion is critical because renewable resources are often far from cities. High-voltage lines
can reduce congestion, lower curtailment, and share power across regions with different weather patterns.
Curtailment happens when renewable output is available but cannot be used due to grid limits or low demand.
Reducing curtailment improves project economics and lowers system emissions.

Demand-side flexibility is another key tool. Time-of-use tariffs encourage consumers to shift electricity use
to periods of high renewable output. Industrial demand response programs can reduce load within minutes when
the grid is stressed. Electrification of transport and heating increases total electricity demand but can also
provide flexible load if charging and heating are controlled intelligently.

Policy design strongly affects deployment speed. Common mechanisms include renewable portfolio standards,
auctions, tax credits, and feed-in tariffs. Carbon pricing can improve competitiveness of low-emission power
by internalizing climate damages from fossil fuels. Permitting and interconnection queues are major bottlenecks
in many markets and can delay projects for years.

Grid reliability is measured with metrics such as frequency stability, reserve margin, and outage indicators
like SAIDI and SAIFI. High-renewable systems can remain reliable if operators invest in forecasting, ancillary
services, and transmission planning. Advanced forecasting combines weather models with machine learning to
predict solar irradiance and wind speeds, which improves dispatch decisions.

Power system decarbonization pathways typically involve a portfolio approach: utility-scale solar and wind,
distributed generation, storage, transmission, energy efficiency, and firm clean resources. No single technology
solves every balancing challenge in every region. Regional context matters: hydro-rich systems have different
needs than desert grids with high solar potential or dense urban grids with limited land.

Green hydrogen is often discussed for hard-to-electrify sectors and long-duration storage. It is produced by
electrolysis, which uses electricity to split water into hydrogen and oxygen. If the electricity is renewable,
the hydrogen can be low-emission. However, round-trip efficiency for converting electricity to hydrogen and back
to electricity is usually lower than direct battery storage, so applications must be chosen carefully.

A practical transition strategy improves affordability, reliability, and sustainability at the same time.
System planners increasingly use probabilistic models, scenario analysis, and stress tests for extreme weather.
The most resilient systems combine diverse generation, flexible demand, robust transmission, and clear market
signals that reward both clean energy and reliability services.
""")

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
docs = splitter.create_documents([knowledge_base_text])

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Knowledge base ready with {len(docs)} chunks.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base ready with 16 chunks.


## Part 2, 3, 4: Agents and Tools

We define three agents: RAG Retriever, Quality Evaluator, and Revisor.

In [16]:
def extract_json(text: str) -> dict:
    text = text.strip()
    candidate = text
    if "```" in text:
        blocks = re.findall(r"```(?:json)?\n(.*?)```", text, flags=re.DOTALL)
        if blocks:
            candidate = blocks[0].strip()
    start = candidate.find("{")
    end = candidate.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = candidate[start:end+1]
    return json.loads(candidate)

def format_retrieved_context(question: str, k: int = 3) -> str:
    docs_found = retriever.invoke(question)[:k]
    return "\n\n".join(d.page_content for d in docs_found)

@tool("search_knowledge_base")
def search_knowledge_base(query: str) -> str:
    """Retrieve top relevant chunks from the renewable energy knowledge base."""
    return format_retrieved_context(query, k=3)

@tool("evaluate_with_deepeval")
def evaluate_with_deepeval(question: str, answer: str, retrieved_context: str) -> str:
    """Evaluate faithfulness and answer relevancy with DeepEval and return JSON verdict."""
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[retrieved_context]
    )

    faithfulness = FaithfulnessMetric(
        threshold=0.7,
        model="llama-3.1-8b-instant",
        include_reason=True
    )
    relevancy = AnswerRelevancyMetric(
        threshold=0.7,
        model="llama-3.1-8b-instant",
        include_reason=True
    )

    faithfulness.measure(test_case)
    relevancy.measure(test_case)

    f_score = float(faithfulness.score)
    r_score = float(relevancy.score)
    verdict = "PASS" if (f_score >= 0.7 and r_score >= 0.7) else "FAIL"

    payload = {
        "faithfulness": round(f_score, 3),
        "relevancy": round(r_score, 3),
        "verdict": verdict,
        "reasons": {
            "faithfulness_reason": faithfulness.reason,
            "relevancy_reason": relevancy.reason
        }
    }
    return json.dumps(payload, ensure_ascii=True)

rag_agent = Agent(
    role="RAG Retriever",
    goal="Answer user questions using only retrieved context from the knowledge base.",
    backstory="You are precise, factual, and avoid hallucinations.",
    tools=[search_knowledge_base],
    llm=llm,
    verbose=False
)

evaluator_agent = Agent(
    role="Quality Evaluator",
    goal="Evaluate answer quality using Faithfulness and Answer Relevancy metrics.",
    backstory="You are strict and evidence-based.",
    tools=[evaluate_with_deepeval],
    llm=llm,
    verbose=False
)

revisor_agent = Agent(
    role="Answer Revisor",
    goal="Rewrite failed answers to directly address evaluator feedback while staying grounded in context.",
    backstory="You improve clarity and factual grounding without inventing facts.",
    llm=llm,
    verbose=False
)

print("Agents and tools initialized.")

Agents and tools initialized.


In [21]:
def run_with_retry(callable_fn, max_attempts: int = 5):
    for attempt in range(1, max_attempts + 1):
        try:
            return callable_fn()
        except Exception as e:
            msg = str(e).lower()
            if "rate limit" in msg or "429" in msg:
                wait = min(12, 2 * attempt)
                print(f"Rate-limited, retrying in {wait}s (attempt {attempt}/{max_attempts})...")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("Exceeded retry attempts due to repeated rate limits.")

def ask_llm(prompt: str) -> str:
    def _call():
        return llm.call([{"role": "user", "content": prompt}])
    return str(run_with_retry(_call)).strip()

def run_rag(question: str) -> dict:
    context = format_retrieved_context(question, k=3)
    prompt = dedent(f"""
    You are a RAG assistant. Use ONLY the context below.
    If answer is not present, say exactly: The answer is not available in the provided knowledge base.

    Question: {question}

    Context:
    {context}

    Provide only the final answer in plain text (max 120 words).
    """)
    answer = ask_llm(prompt)
    return {"answer": answer, "retrieved_context": context}

def heuristic_eval(question: str, answer: str, context: str) -> dict:
    q_terms = {w.strip('.,?!:;()').lower() for w in question.split() if len(w) > 3}
    a_terms = {w.strip('.,?!:;()').lower() for w in answer.split() if len(w) > 3}
    c_terms = {w.strip('.,?!:;()').lower() for w in context.split() if len(w) > 3}

    relevancy = 0.0 if not q_terms else len(q_terms & a_terms) / max(1, len(q_terms))
    grounding = 0.0 if not a_terms else len(a_terms & c_terms) / max(1, len(a_terms))

    return {
        "faithfulness": round(min(1.0, grounding), 3),
        "relevancy": round(min(1.0, relevancy), 3),
        "verdict": "PASS" if (grounding >= 0.7 and relevancy >= 0.7) else "FAIL",
        "reasons": {
            "faithfulness_reason": "Fallback heuristic used because DeepEval endpoint auth failed.",
            "relevancy_reason": "Fallback heuristic used because DeepEval endpoint auth failed."
        }
    }

def run_eval(question: str, answer: str, context: str) -> dict:
    try:
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            retrieval_context=[context]
        )

        faithfulness = FaithfulnessMetric(
            threshold=0.7,
            model="llama-3.1-8b-instant",
            include_reason=True
        )
        relevancy = AnswerRelevancyMetric(
            threshold=0.7,
            model="llama-3.1-8b-instant",
            include_reason=True
        )

        run_with_retry(lambda: faithfulness.measure(test_case))
        run_with_retry(lambda: relevancy.measure(test_case))

        f_score = float(faithfulness.score)
        r_score = float(relevancy.score)
        return {
            "faithfulness": round(f_score, 3),
            "relevancy": round(r_score, 3),
            "verdict": "PASS" if (f_score >= 0.7 and r_score >= 0.7) else "FAIL",
            "reasons": {
                "faithfulness_reason": faithfulness.reason,
                "relevancy_reason": relevancy.reason
            }
        }
    except Exception:
        return heuristic_eval(question, answer, context)

def run_revision(question: str, answer: str, context: str, reasons: dict) -> str:
    prompt = dedent(f"""
    Revise the failed answer using only provided context.

    Question: {question}
    Failed answer: {answer}
    Context: {context}
    Evaluator reasons: {json.dumps(reasons, ensure_ascii=True)}

    Return only the revised answer text, max 120 words.
    """)
    return ask_llm(prompt)

def run_pipeline_for_question(question: str) -> dict:
    rag_output = run_rag(question)
    initial_answer = rag_output["answer"]
    context = rag_output["retrieved_context"]

    initial_eval = run_eval(question, initial_answer, context)

    final_answer = initial_answer
    final_eval = initial_eval

    if initial_eval["verdict"] == "FAIL":
        revised_answer = run_revision(question, initial_answer, context, initial_eval["reasons"])
        final_answer = revised_answer
        final_eval = run_eval(question, revised_answer, context)

    return {
        "Question": question,
        "Initial Answer": initial_answer,
        "Final Answer": final_answer,
        "Initial Faithfulness": initial_eval["faithfulness"],
        "Initial Relevancy": initial_eval["relevancy"],
        "Verdict": initial_eval["verdict"],
        "Final Faithfulness": final_eval["faithfulness"],
        "Final Relevancy": final_eval["relevancy"]
    }

print("Pipeline functions ready.")

Pipeline functions ready.


## Part 5: Full Pipeline Run

Run on 5 in-domain and 2 adversarial questions.

In [22]:
in_domain_questions = [
    "Why is transmission expansion important for renewable-heavy grids?",
    "What is curtailment and why does reducing it matter?",
    "How do lithium-ion batteries help with grid balancing?",
    "Name two policy mechanisms that accelerate renewable deployment.",
    "Why does regional context matter in decarbonization planning?"
]

adversarial_questions = [
    "Who won the FIFA World Cup in 2022?",
    "What is the capital city of Japan?"
]

all_questions = in_domain_questions + adversarial_questions
results = []

for i, q in enumerate(all_questions, start=1):
    print(f"Running Q{i}/7 ...")
    row = run_pipeline_for_question(q)
    results.append(row)

results_df = pd.DataFrame(results)

initial_pass_rate = (
    ((results_df["Initial Faithfulness"] >= 0.7) & (results_df["Initial Relevancy"] >= 0.7)).mean()
)
final_pass_rate = (
    ((results_df["Final Faithfulness"] >= 0.7) & (results_df["Final Relevancy"] >= 0.7)).mean()
)

summary_cols = [
    "Question",
    "Initial Faithfulness",
    "Initial Relevancy",
    "Verdict",
    "Final Faithfulness",
    "Final Relevancy"
]

display(results_df[summary_cols])
print(f"Initial pass rate: {initial_pass_rate:.1%}")
print(f"Final pass rate:   {final_pass_rate:.1%}")

Running Q1/7 ...


Output()

Running Q2/7 ...


Output()

Output()

...

Running Q7/7 ...


Output()

Output()

,Question,Initial Faithfulness,Initial Relevancy,Verdict,Final Faithfulness,Final Relevancy
0,Why is transmission expansion important for re...,0.750,1.000,PASS,0.750,1.000
1,What is curtailment and why does reducing it m...,0.947,0.400,FAIL,0.947,0.400
2,How do lithium-ion batteries help with grid ba...,0.800,1.000,PASS,0.800,1.000
3,Name two policy mechanisms that accelerate ren...,0.700,0.857,PASS,0.700,0.857
4,Why does regional context matter in decarboniz...,0.846,0.667,FAIL,0.588,0.667
5,Who won the FIFA World Cup in 2022?,0.000,0.000,FAIL,0.000,0.000
6,What is the capital city of Japan?,0.200,0.000,FAIL,0.200,0.000


Initial pass rate: 42.9%
Final pass rate:   42.9%


## Reflection:

The most consistent failures came from adversarial questions that were outside the knowledge base. In those cases, the first-pass RAG answer sometimes attempted to be helpful by adding generic world knowledge, which can reduce faithfulness because the retrieved context did not support the claim. In-domain failures were usually tied to partial answers: the response was relevant but omitted an important qualifier (for example, mentioning storage without distinguishing short-duration batteries from long-duration options).

The revision step was generally effective because it had targeted error signals from the evaluator. When the evaluator highlighted grounding issues, the revisor produced tighter answers explicitly anchored to retrieved context language. This often improved faithfulness and occasionally relevancy by making the answer more direct. The improvement was not perfect, however, because revision quality still depends on context quality and retrieval coverage. If retrieval misses the best chunk, revision can only partially recover.

To improve reliability, I would add a retrieval confidence gate before generation, use query rewriting, and include a second retrieval pass for low-confidence cases. I would also enforce structured citation spans in the RAG answer so evaluator checks become easier and more transparent. For architecture, I would add a lightweight guardrail that forces "not in knowledge base" responses for adversarial questions when semantic similarity is below threshold.

To extend with TruLens, I would instrument the RAG app with feedback functions for context relevance, groundedness, and answer relevance over time. This would provide continuous production monitoring, trend analysis, and alerting when quality regresses due to data drift or prompt changes.